# Method (E) — GeoCP-only baseline

Post-hoc computation of the reviewer-requested baseline: **raw APS + per-pixel Gaussian-weighted quantile** (no SACP smoothing). Reads the existing α-sweep pkls from Drive, runs 1-D CV on bw only, then evaluates at α ∈ {0.05, 0.10, 0.15} on each run.

No model retraining needed. CPU-only. Estimated runtime: 20–60 min total on a Colab CPU runtime.

Outputs written to Drive:
- `/content/drive/MyDrive/method_E/method_E_per_run.csv`
- `/content/drive/MyDrive/method_E/method_E_paired_tests.csv`

Atomic writes + per-run caching so you can resume if the runtime dies.

## 1. Mount Drive + paths

In [1]:
!pip install -q scikit-learn scipy pandas
import os, time, pickle, shutil, json
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

HSI_CKPT = '/content/drive/MyDrive/hsi_alpha_sweep/checkpoints'
S2_CKPT  = '/content/drive/MyDrive/s2_alpha_sweep/checkpoints'

LOCAL_DIR  = '/content/method_E'
RUN_CACHE  = f'{LOCAL_DIR}/runs'  # per-run intermediate pkls
OUT_DIR    = f'{LOCAL_DIR}/out'
for d in [LOCAL_DIR, RUN_CACHE, OUT_DIR]: os.makedirs(d, exist_ok=True)

DRIVE_OUT  = '/content/drive/MyDrive/method_E'
try: os.makedirs(DRIVE_OUT, exist_ok=True)
except OSError: pass

# Restore any cached per-run results from Drive to local
def _restore():
    src = f'{DRIVE_OUT}/runs'
    if not os.path.isdir(src): return 0
    n = 0
    for f in os.listdir(src):
        if not f.endswith('.pkl'): continue
        dst = f'{RUN_CACHE}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(f'{src}/{f}', dst); n += 1
        except OSError as e: print(f'[restore] {f}: {e}')
    return n
print(f'[restore] copied {_restore()} cached run results from Drive')
print(f'HSI_CKPT : {HSI_CKPT}  ({len(os.listdir(HSI_CKPT))} pkls)')
print(f'S2_CKPT  : {S2_CKPT}   ({len(os.listdir(S2_CKPT))} pkls)')

Mounted at /content/drive
[restore] copied 0 cached run results from Drive
HSI_CKPT : /content/drive/MyDrive/hsi_alpha_sweep/checkpoints  (50 pkls)
S2_CKPT  : /content/drive/MyDrive/s2_alpha_sweep/checkpoints   (29 pkls)


## 2. Helpers

In [2]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.model_selection import KFold
from scipy import stats as sstats

BW_GRID    = [3, 5, 7, 10, 15, 20, 30, 50, 100]
ALPHA_GRID = (0.05, 0.10, 0.15)
ALPHA_CV   = 0.10
CV_FOLDS   = 5
MAX_CAL_S2 = 20000

def vwq(sorted_scores, d_mat, order, bw, alpha):
    log_w = -0.5 * (d_mat / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def eval_set(score_mat, q_vec, y_true, alpha):
    in_set = score_mat < q_vec[:, None]
    sizes = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    IS = float((sizes + (2.0/alpha) * (~covered).astype(np.float64)).mean())
    size_mean = float(sizes.mean()); cov_mean = float(covered.mean())
    buckets = [(1,1),(2,2),(3,3),(4,4),(5,10**9)]
    worst = 0.0
    for lo, hi in buckets:
        m = (sizes >= lo) & (sizes <= hi)
        if m.sum() == 0: continue
        c = float(covered[m].mean()); worst = max(worst, abs(c - (1 - alpha)))
    return IS, cov_mean, size_mean, worst * 100

def stratified_sub(labels, n, seed):
    if n is None or n >= len(labels): return np.arange(len(labels))
    uniq, cnts = np.unique(labels, return_counts=True)
    per_cls_n = np.maximum(1, (n * cnts / cnts.sum()).astype(int))
    rng = np.random.RandomState(seed); out = []
    for c, nc in zip(uniq, per_cls_n):
        pool = np.where(labels == c)[0]
        out.append(pool if len(pool) <= nc else rng.choice(pool, size=nc, replace=False))
    return np.concatenate(out)

def run_method_E_one(d, experiment):
    H = int(d.get('H', d.get('h'))); W = int(d.get('W', d.get('w')))
    cal_true = np.asarray(d['cal_true_aps'])
    cal_all  = np.asarray(d['cal_all_aps'])
    test_all = np.asarray(d['test_all_aps'])
    cal_idx  = np.asarray(d.get('cal_flat_idx', d.get('ca_gi')))
    te_idx   = np.asarray(d.get('test_flat_idx', d.get('te_gi')))
    y_ca     = np.asarray(d['y_ca']); y_te = np.asarray(d['y_te'])

    if experiment == 'S2':
        sub = stratified_sub(y_ca, MAX_CAL_S2, 0*100 + 43)
    else:
        sub = np.arange(len(cal_idx))
    cal_true_sub = cal_true[sub]; cal_all_sub = cal_all[sub]
    cal_idx_sub  = cal_idx[sub]; y_ca_sub = y_ca[sub]
    n_sub = len(sub)
    coords_sub = np.stack([cal_idx_sub // W, cal_idx_sub % W], 1).astype(float)
    coords_te  = np.stack([te_idx // W, te_idx % W], 1).astype(float)

    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=0)
    cv_is = {bw: [] for bw in BW_GRID}
    for f_tr, f_val in kf.split(np.arange(n_sub)):
        tr_scores = cal_true_sub[f_tr]; val_scores = cal_all_sub[f_val]; val_y = y_ca_sub[f_val]
        order_tr = np.argsort(tr_scores); sorted_tr = tr_scores[order_tr]
        BATCH = 500
        for bw in BW_GRID:
            q_val = np.empty(len(f_val))
            for b0 in range(0, len(f_val), BATCH):
                b1 = min(b0 + BATCH, len(f_val))
                dcv = cdist(coords_sub[f_val[b0:b1]], coords_sub[f_tr])
                q_val[b0:b1] = vwq(sorted_tr, dcv, order_tr, bw, ALPHA_CV)
            IS_val, _, _, _ = eval_set(val_scores, q_val, val_y, ALPHA_CV)
            cv_is[bw].append(IS_val)
    cv_mean = {bw: float(np.mean(v)) for bw, v in cv_is.items()}
    best_bw = int(min(cv_mean, key=cv_mean.get))

    order = np.argsort(cal_true_sub); sorted_cal = cal_true_sub[order]
    n_te = len(te_idx); BATCH_TEST = 1000
    out_per_alpha = {}
    q_cache = {a: np.empty(n_te) for a in ALPHA_GRID}
    for b0 in range(0, n_te, BATCH_TEST):
        b1 = min(b0 + BATCH_TEST, n_te)
        dtest = cdist(coords_te[b0:b1], coords_sub)
        for a in ALPHA_GRID:
            q_cache[a][b0:b1] = vwq(sorted_cal, dtest, order, best_bw, a)
    for a in ALPHA_GRID:
        IS, cov, sz, sscv = eval_set(test_all, q_cache[a], y_te, a)
        out_per_alpha[a] = {'is': IS, 'cov': cov, 'size': sz, 'sscv_pct': sscv}
    return best_bw, out_per_alpha, cv_mean

print('Helpers loaded.')

Helpers loaded.


## 3. Run method E on all 79 runs (resumable)

In [ ]:
def _mirror_run(fname):
    src = f'{RUN_CACHE}/{fname}'; dst = f'{DRIVE_OUT}/runs/{fname}'
    try:
        os.makedirs(f'{DRIVE_OUT}/runs', exist_ok=True); shutil.copy2(src, dst); return True
    except OSError: return False

def process(pkl_path, experiment):
    key = os.path.basename(pkl_path).replace('.pkl', '')
    cache_file = f'{RUN_CACHE}/{experiment}_{key}.pkl'
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f: return pickle.load(f)
    with open(pkl_path, 'rb') as f: d = pickle.load(f)
    if d.get('degenerate'): return None
    t0 = time.time()
    best_bw, per_a, cv_mean = run_method_E_one(d, experiment)
    result = {'experiment': experiment, 'key': key, 'best_bw': best_bw,
              'per_alpha': per_a, 'cv_is_mean': cv_mean,
              'n_test': int(len(d['y_te'])), 'dt': time.time() - t0}
    if experiment == 'HSI':
        result['dataset'] = d['dataset']; result['seed'] = int(d['seed'])
        result['config_id'] = f'{d["dataset"]}_seed{d["seed"]}'
    else:
        result['tile'] = d['tile']; result['size_px'] = int(d['size_px'])
        result['config_id'] = f'{d["tile"]}_s{d["size_px"]}'
    with open(cache_file, 'wb') as f: pickle.dump(result, f)
    _mirror_run(os.path.basename(cache_file))
    return result

runs = []
t_start = time.time()
print('=' * 60); print('HSI (50 runs)'); print('=' * 60)
for i, f in enumerate(sorted(os.listdir(HSI_CKPT))):
    if not f.endswith('.pkl'): continue
    r = process(f'{HSI_CKPT}/{f}', 'HSI')
    if r is None: continue
    runs.append(r)
    pa = r['per_alpha']
    print(f'  [{i+1:>2}] {f}  bw*={r["best_bw"]:>3}  IS@0.10={pa[0.10]["is"]:.3f}  cov@0.10={pa[0.10]["cov"]:.3f}  ({r["dt"]:.0f}s)')

print('\n' + '=' * 60); print('S2 (29 runs)'); print('=' * 60)
for i, f in enumerate(sorted(os.listdir(S2_CKPT))):
    if not f.endswith('.pkl'): continue
    r = process(f'{S2_CKPT}/{f}', 'S2')
    if r is None:
        print(f'  [{i+1:>2}] {f}  SKIP (degenerate)'); continue
    runs.append(r)
    pa = r['per_alpha']
    print(f'  [{i+1:>2}] {f}  bw*={r["best_bw"]:>3}  IS@0.10={pa[0.10]["is"]:.3f}  cov@0.10={pa[0.10]["cov"]:.3f}  ({r["dt"]:.0f}s)')

print(f'\n==== ALL DONE: {len(runs)} runs in {(time.time()-t_start)/60:.1f} min ====')

HSI (50 runs)
  [ 1] botswana_seed0.pkl  bw*= 50  IS@0.10=2.980  cov@0.10=0.900  (1s)
  [ 2] botswana_seed1.pkl  bw*= 50  IS@0.10=2.853  cov@0.10=0.911  (1s)
  [ 3] botswana_seed2.pkl  bw*=100  IS@0.10=2.994  cov@0.10=0.900  (1s)
  [ 4] botswana_seed3.pkl  bw*=100  IS@0.10=3.042  cov@0.10=0.897  (1s)
  [ 5] botswana_seed4.pkl  bw*=100  IS@0.10=2.700  cov@0.10=0.914  (1s)
  [ 6] botswana_seed5.pkl  bw*= 50  IS@0.10=2.833  cov@0.10=0.907  (1s)
  [ 7] botswana_seed6.pkl  bw*= 50  IS@0.10=2.933  cov@0.10=0.904  (1s)
  [ 8] botswana_seed7.pkl  bw*= 50  IS@0.10=2.969  cov@0.10=0.901  (1s)
  [ 9] botswana_seed8.pkl  bw*=100  IS@0.10=2.903  cov@0.10=0.904  (1s)
  [10] botswana_seed9.pkl  bw*=100  IS@0.10=3.043  cov@0.10=0.897  (1s)
  [11] ip_seed0.pkl  bw*= 10  IS@0.10=4.058  cov@0.10=0.906  (11s)
  [12] ip_seed1.pkl  bw*= 10  IS@0.10=4.153  cov@0.10=0.911  (11s)
  [13] ip_seed2.pkl  bw*= 15  IS@0.10=4.296  cov@0.10=0.911  (11s)
  [14] ip_seed3.pkl  bw*= 20  IS@0.10=3.959  cov@0.10=0.916  (11s

## 4. Aggregate per-run results + paired tests

In [ ]:
# Per-run CSV
rows = []
for r in runs:
    for a, m in r['per_alpha'].items():
        base = {'experiment': r['experiment'], 'config_id': r['config_id'],
                'alpha': a, 'method': 'E', 'best_bw': r['best_bw'], **m}
        if r['experiment'] == 'HSI':
            base['dataset'] = r['dataset']; base['seed'] = r['seed']
        else:
            base['tile'] = r['tile']; base['size_px'] = r['size_px']
        rows.append(base)
df_E = pd.DataFrame(rows)
df_E.to_csv(f'{OUT_DIR}/method_E_per_run.csv', index=False)
print(f'Wrote method_E_per_run.csv ({len(df_E)} rows)')
print('\n=== Method E pooled ===')
pooled = (df_E.groupby(['experiment','alpha'])
                .agg(is_mean=('is','mean'), cov_mean=('cov','mean'),
                     size_mean=('size','mean'), sscv_mean=('sscv_pct','mean'),
                     n=('alpha','count')).reset_index())
print(pooled.round(3).to_string(index=False))

Wrote method_E_per_run.csv (237 rows)

=== Method E pooled ===
experiment  alpha  is_mean  cov_mean  size_mean  sscv_mean  n
       HSI   0.05    3.686     0.953      1.803      7.035 50
       HSI   0.10    3.384     0.904      1.462      9.976 50
       HSI   0.15    3.257     0.852      1.287     17.252 50
        S2   0.05    3.594     0.948      1.530      5.239 29
        S2   0.10    3.343     0.898      1.303      9.726 29
        S2   0.15    3.206     0.846      1.158     13.829 29


In [ ]:
# Paired tests against D / A / B / C from existing analysis CSVs.
# Upload these CSVs to Drive if not present already, or let the user copy them.
ANAL = '/content/drive/MyDrive/GeoCP_RS_analysis'   # where per_seed_alpha.csv + s2_per_config_alpha.csv live
# If you keep them in the repo, update this path to the local copy you've uploaded to Drive.
HSI_MAIN = f'{ANAL}/per_seed_alpha.csv'
S2_MAIN  = f'{ANAL}/s2_per_config_alpha.csv'

if not (os.path.exists(HSI_MAIN) and os.path.exists(S2_MAIN)):
    print('Skipping paired tests — upload per_seed_alpha.csv and s2_per_config_alpha.csv to', ANAL)
    print('(or change the paths above). Per-run CSV is already written and can be analysed locally.')
else:
    df_hsi = pd.read_csv(HSI_MAIN)
    df_s2  = pd.read_csv(S2_MAIN)
    df_hsi['config_id'] = df_hsi['dataset'] + '_seed' + df_hsi['seed'].astype(str)
    df_s2['config_id']  = df_s2['tile']     + '_s'     + df_s2['size_px'].astype(str)
    df_main_all = pd.concat([df_hsi.assign(experiment='HSI'),
                             df_s2.assign(experiment='S2')], ignore_index=True)

    results = []
    for exp, df_main in [('HSI', df_hsi), ('S2', df_s2), ('GRAND', df_main_all)]:
        if exp == 'GRAND':
            df_E_exp = df_E
        else:
            df_E_exp = df_E[df_E.experiment == exp]
        for a in ALPHA_GRID:
            sub_E = df_E_exp[df_E_exp.alpha == a].sort_values(['experiment','config_id']).reset_index(drop=True) if exp == 'GRAND' else df_E_exp[df_E_exp.alpha == a].sort_values('config_id').reset_index(drop=True)
            for ref in ('D','A','B','C'):
                if exp == 'GRAND':
                    sub_R = df_main_all[(df_main_all.alpha == a) & (df_main_all.method == ref)].sort_values(['experiment','config_id']).reset_index(drop=True)
                else:
                    sub_R = df_main[(df_main.alpha == a) & (df_main.method == ref)].sort_values('config_id').reset_index(drop=True)
                if len(sub_R) != len(sub_E):
                    print(f'[WARN] {exp} α={a} vs {ref}: len mismatch {len(sub_E)} vs {len(sub_R)}'); continue
                assert (sub_R.config_id.values == sub_E.config_id.values).all()
                impr = (sub_R['is'].values - sub_E['is'].values) / sub_R['is'].values * 100
                t, p = sstats.ttest_1samp(impr, 0.0)
                results.append({'experiment': exp, 'alpha': a, 'comparison': f'E vs {ref}',
                                'mean_pct': float(impr.mean()),
                                'sem': float(impr.std(ddof=1)/np.sqrt(len(impr))),
                                't': float(t), 'p': float(p), 'n': int(len(impr))})
    df_tests = pd.DataFrame(results)
    df_tests.to_csv(f'{OUT_DIR}/method_E_paired_tests.csv', index=False)
    print(f'Wrote method_E_paired_tests.csv ({len(df_tests)} rows)')
    print('\n=== E vs {D, A, B, C} ===')
    print(df_tests.round(4).to_string(index=False))

Skipping paired tests — upload per_seed_alpha.csv and s2_per_config_alpha.csv to /content/drive/MyDrive/GeoCP_RS_analysis
(or change the paths above). Per-run CSV is already written and can be analysed locally.


## 5. Push outputs to Drive

In [ ]:
for f in os.listdir(OUT_DIR):
    try: shutil.copy2(f'{OUT_DIR}/{f}', f'{DRIVE_OUT}/{f}')
    except OSError as e: print(f'Drive copy failed for {f}: {e}')
print('Outputs pushed to', DRIVE_OUT)

Outputs pushed to /content/drive/MyDrive/method_E
